<a href="https://colab.research.google.com/github/Viggo-Kristensen/kaggle-competitions/blob/main/f1-pit-stop-prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Problem Definition**

### Predict whether a f1 car is going in for a pit stop on a given lap.

## **2. Imports**

In [29]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

# Metrics
from sklearn.metrics import roc_auc_score



## **3. Dataset**

Each row represents a specific car on a given lap. The target variable `PitNextLap` indicates whether a car will pit on the following lap. Features include `Lap_Timing`, `RaceProgress`, `Tyre_state` and positional information.

### Uploading Files

In [6]:
!mkdir -p /root/.kaggle
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [9]:
!pip install -q kaggle

# upload kaggle.json ONCE
from google.colab import files
files.upload()

!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c playground-series-s6e5

Saving kaggle.json to kaggle.json
100% 19.2M/19.2M [00:00<00:00, 190MB/s]



### Unzipping Files

In [10]:
!unzip -q playground-series-s6e5.zip -d data

### Creating DataFrames

In [47]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [48]:
target = "PitNextLap"

drop_cols = ["id", "Driver", target]

X = train.drop(columns=drop_cols)
y = train[target]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "bool"]).columns

## **4. Feature Engineering**

In [49]:
class AddFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["Pit_Window_Signal"] = X["LapNumber"] / (X["TyreLife"] + 1e-6)
        X["Laps_Left"] = (1 - X["RaceProgress"]) / (X["RaceProgress"] / X["LapNumber"])
        X["Total_Laps"] = 1 / (X["RaceProgress"] / X["LapNumber"])

        return X

## **5. Preprocessing**

### Numeric Pipeline

In [50]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

### Categorical Pipeline

In [51]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

### Preprocessor

In [52]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ],
    remainder="passthrough"
)

## **6. Pipeline**

In [53]:
model = Pipeline(steps=[
    ("feature_engineering", AddFeatures()),
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

## **7. Cross Validation**

In [46]:
scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc", n_jobs=-1)

print("fold scores:", scores)
print("mean roc-auc", scores.mean())

fold scores: [0.85950311 0.86213958 0.8615101  0.86020309 0.86123367]
mean roc-auc 0.8609179113032193


## **8. Training**

In [54]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppr

Pipeline(steps=[('feature_engineering', AddFeatures()),
                ('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Compound', 'Race'], dtype='object'))])),
                ('classifier', LogisticRegression(max_iter=1000))])

## **9. Feature Importance Analysis**

In [59]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

coefficients = model.named_steps["classifier"].coef_[0]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

importance_df["Abs_Coefficient"] = importance_df["Coefficient"].abs()

importance_df = importance_df.sort_values(
    by="Abs_Coefficient",
    ascending=False
)

print(importance_df.head(100))

                                Feature  Coefficient  Abs_Coefficient
2                        num__LapNumber     4.881604         4.881604
9                     num__RaceProgress    -4.595453         4.595453
15                    cat__Compound_WET    -1.656699         1.656699
32     cat__Race_Mexico City Grand Prix    -1.278317         1.278317
21         cat__Race_Belgian Grand Prix     1.110498         1.110498
12           cat__Compound_INTERMEDIATE    -1.045152         1.045152
23        cat__Race_Canadian Grand Prix    -1.014461         1.014461
13                 cat__Compound_MEDIUM    -0.809560         0.809560
33           cat__Race_Miami Grand Prix    -0.789220         0.789220
25           cat__Race_Dutch Grand Prix    -0.745219         0.745219
20         cat__Race_Bahrain Grand Prix     0.744603         0.744603
17      cat__Race_Australian Grand Prix    -0.718875         0.718875
39         cat__Race_Spanish Grand Prix     0.706443         0.706443
18        cat__Race_

## **10. Evaluation**

In [55]:
val_probs = model.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, val_probs)

np.float64(0.8615262111019975)

## **11. Submission**

In [60]:
submission = pd.read_csv("data/sample_submission.csv")
submission["PitNextLap"] = model.predict_proba(test)[:, 1]
submission.to_csv("submission.csv", index=False)

## **12. Reflections**

**Notes**


*   Dropped the Driver category since the weights were all near 0

*   Among the engineered features `Laps_Left` proved to be the strongest predictor.

*   The final model achieved a public leaderboard ROC-AUC score of 0.885, placing #840 on the Kaggle leaderboard at submission time.




**What i learned**


*   Building end to end pipelines using scikit-learn


*   Using cross validation to evaluate models more reliably


*   Evaluating the usability of specific features using coefficients

*   Understanding the effect of L2 regularisation

*   Designing high impact features for linear models








